## **ENTRENAMIENTO SAM 2.1 CON KVASIR-SEG**

En este fichero se evalúa el rendimiento de los modleos SAM 2.1 Base y SAM 2.1 Large tras haberlos entrenado usando el dataset KVASIR-SEG.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [ ]:
from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_fine_tuning
from torch.utils.data import Dataset, DataLoader
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import numpy as np
import os
import cv2
import pandas as pd
import json
import time
import sys
import torch
import shutil
import random
import torch.nn as nn


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"


La siguiente función divide el conjunto de datos de imágenes en entrenamiento, validación y test (70, 15, 15) creando las subcarpetas correspondientes tanto para imágenes como para máscaras, garantizando la correspondencia entre ambas. Para ello, se establece una semilla, evitando así crear conjuntos diferentes cada vez que se ejecute la función. 

In [ ]:
def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "masks")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".jpg"), os.path.join(dataset, "masks",  split, name + ".jpg"))

split_dataset(dataset)


En primer lugar, se habilita TF32 para las operaciones matriciales, ya que esto nos permite acelerar el entrenamiento en GPUs NVIDIA modernas con una pérdida de precisión mínima. Además, se define el dispositivo donde se ejecutará el entrenamiento y se comprueba la disponibilidad de CUDA.

A continuación, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test), junto con el fichero json que contiene las cajas delimitadoras de cada imagen.

Por último, en el método __getitem__ se aplica el preprocesamiento de cada muestra. La imagen se redimensiona a 1024x1024, se normaliza entre 0 y 1 y se convierte a tensor. Por otro lado, la máscara se redimensiona a 256x256 (resolución de salida del mask decoder de SAM) y se binariza. Finalmente, a partir de la caja delimitadora se calcula el punto central escalado a las nuevas dimensiones de la imagen, que es el prompt que se le pasaría al modelo.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, bbox_json, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        with open(bbox_json) as f:
            bbox_json = json.load(f)

        for img_name, info in bbox_json.items():
            img_path  = os.path.join(self.images_dir, img_name + ".jpg")
            mask_path = os.path.join(self.masks_dir,  img_name + ".jpg")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.samples.append((img_path, mask_path, info["bbox"][0]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, bbox = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (256, 256))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
        point = torch.tensor([[xmin + (xmax-xmin)/2, ymin + (ymax-ymin)/2]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


Esta función contiene el bucle de entrenamiento. En primer lugar, se carga el modelo SAM 2 mediante build_sam2, que requiere además de los pesos un fichero de configuración específico de la arquitectura. Al igual que con SAM 1, se congelan tanto el image encoder como el prompt encoder, de forma que durante el entrenamiento solo se actualizan los pesos del mask decoder.

Como optimizador se usa Adam con un learning rate de 1e-4, y se añade un scheduler que reduce el learning rate a la mitad cada 20 épocas, ayudando así a afinar el modelo conforme avanza el entrenamiento. Además, se usa BCEWithLogitsLoss como función de pérdida, ya que es adecuada para segmentación binaria.

Por último, al usar la API de SAM 2, el image encoder se ejecuta a través de predictor.set_image, lo que permite acceder a las características de alta resolución (high_res_feats) que SAM 2 incorpora y que se pasan directamente al mask decoder. Finalmente, el guardado de pesos se hace en formato de diccionario con la clave "model", ya que es el formato esperado por la API de SAM 2.

In [ ]:
def train_sam(dataset_path, bbox_json, weights_path, config_path, output_weights, epochs=50, batch_size=4):
    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)

    for param in sam2.image_encoder.parameters():
        param.requires_grad = False
    for param in sam2.sam_prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam2.sam_mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train", bbox_json)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam2.train()
    predictor = SAM2ImagePredictor(sam2)

    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            masks  = masks.to(device)
            points = points.to(device)
            labels = labels.to(device)

            loss_batch = 0
            for i in range(images.shape[0]):
                image_np = (images[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)

                with torch.no_grad():
                    predictor.set_image(image_np)

                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam2.sam_prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                image_embedding = predictor._features["image_embed"]
                image_pe        = sam2.sam_prompt_encoder.get_dense_pe()
                high_res_features = predictor._features["high_res_feats"]

                low_res_masks, _, _, _ = sam2.sam_mask_decoder(
                    image_embeddings=image_embedding,
                    image_pe=image_pe,
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                    repeat_image=False,
                    high_res_features=high_res_features,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save({"model": sam2.state_dict()}, output_weights)
    return output_weights


**SAM 2.1 BASE ENTRENAMIENTO**

En primer lugar, se definen las rutas a los distintos recursos que se van a usar: el dataset, el fichero de cajas delimitadoras, los pesos preentrenados, el fichero de configuración de la arquitectura y las rutas donde se guardarán los resultados y los pesos del modelo entrenado. En este caso, se procede a evaluar el rendimiento del modelo SAM 2.1 Base.

Respecto a la función de evaluación, esta función evalúa el modelo fine-tuneado sobre el conjunto de test. Para cada imagen se carga su máscara, se calcula el punto central a partir de la caja delimitadora y se realiza la inferencia. La máscara predicha se redimensiona al tamaño original de la imagen para que sea comparable con el ground truth, y se calculan todas las métricas. Al final se devuelve un diccionario con la media de cada métrica sobre todo el conjunto de test.

Finalmente, se lanza el entrenamiento midiendo el tiempo que tarda, y después se evalúa el modelo resultante, midiendo también su tiempo. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guarda en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores, lo que permite ir acumulando los resultados de distintos experimentos en el mismo fichero.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
config_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2.1\\sam2.1_hiera_b+.yaml"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2_1b_kvasir.pt"


def evaluate_model(dataset, weights_path, config_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference_fine_tuning(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_2.1_b_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results


start_train = time.time()
trained_weights = train_sam(dataset, bboxes, weights, config_path, output_weights)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights, config_path)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.7184
Mean F1 Score: 0.8039
Mean Recall: 0.8060
Mean Precision: 0.8663
Mean Dice: 0.8039
Mean Specificity: 0.9829
Mean F2: 0.7994
Mean Pixel Accuracy: 0.9405
Mean Boundary IoU: 0.0928
Mean Hausdorff-95: 133.0716
Mean Latency (ms): 227.55
Mean VRAM Usage (MB): 0.11


**SAM 2.1 LARGE ENTRENAMIENTO**

A continuación, se replica el procedimiento aplicado en el bloque de código anterior con la única diferencia de que el modelo a evaluar en este caso será SAM 2.1 Large. De esta manera podremos comparar las diferencias de rendimiento entre las distintas versiones de cada modelo.

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"
config_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\sam2\\sam2\\configs\\sam2.1\\sam2.1_hiera_l.yaml"
output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_sam2_1l_kvasir.pt"


def evaluate_model(dataset, weights_path, config_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam2 = build_sam2(config_path, weights_path)
    sam2.to(device)
    sam2.eval()
    predictor = SAM2ImagePredictor(sam2)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference_fine_tuning(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_2.1_l_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results


start_train = time.time()
trained_weights = train_sam(dataset, bboxes, weights, config_path, output_weights)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_model(dataset, trained_weights, config_path)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Mean IoU: 0.7685
Mean F1 Score: 0.8426
Mean Recall: 0.8623
Mean Precision: 0.8725
Mean Dice: 0.8426
Mean Specificity: 0.9800
Mean F2: 0.8497
Mean Pixel Accuracy: 0.9495
Mean Boundary IoU: 0.1090
Mean Hausdorff-95: 122.1043
Mean Latency (ms): 292.41
Mean VRAM Usage (MB): 0.11
